# 🤖 Notebook 04 — Full Agent Loop with OpenAI SDK

**What you'll learn:**
- Wire ctxtual into a real OpenAI function-calling loop
- Use the integration adapters (`to_openai_tools`, `handle_tool_calls`)
- Watch the agent autonomously explore data across multiple turns
- See how system_prompt + tool schemas + WorkspaceRef guide the LLM

**Two modes:**
1. **Simulated** (no API key) — uses a scripted sequence to demonstrate the flow
2. **Live** (with `OPENAI_API_KEY`) — real GPT-4o agent

> This notebook runs in simulated mode by default. Set `USE_REAL_API = True` and provide your key to use GPT-4o.

In [ ]:
USE_REAL_API = False  # Set True + provide OPENAI_API_KEY to use real GPT-4o
# import os; os.environ["OPENAI_API_KEY"] = "sk-..."  # Uncomment if using real API

## 1. Setup: Ctx + Producer + Tools

In [ ]:
import json
from ctxtual import Ctx, MemoryStore
from ctxtual.utils import paginator, text_search, filter_set, pipeline

ctx = Ctx(store=MemoryStore())
pager = paginator(ctx, "tickets")
search = text_search(ctx, "tickets", fields=["subject", "body"])
filters = filter_set(ctx, "tickets")
pipe = pipeline(ctx, "tickets")

import random
random.seed(42)

urgencies = ["low", "medium", "high", "critical"]
categories = ["billing", "technical", "account", "feature_request", "bug_report"]
agents_list = ["Alice", "Bob", "Carol", "Dave", "Eve"]

@ctx.producer(workspace_type="tickets", toolsets=[pager, search, filters, pipe])
def load_support_tickets() -> list[dict]:
    """Load all open support tickets from the helpdesk system."""
    tickets = []
    for i in range(2000):
        tickets.append({
            "ticket_id": f"TKT-{i:04d}",
            "subject": f"Issue with {random.choice(['login', 'payment', 'sync', 'export', 'API', 'dashboard', 'mobile app', 'email', 'SSO', 'webhook'])} - case {i}",
            "body": f"Customer reports {random.choice(['error', 'timeout', 'crash', 'incorrect data', 'slow performance', 'missing feature'])} when {random.choice(['logging in', 'processing payment', 'syncing data', 'exporting CSV', 'calling API'])}.",
            "urgency": random.choice(urgencies),
            "category": random.choice(categories),
            "assigned_to": random.choice(agents_list),
            "created_days_ago": random.randint(0, 90),
            "response_count": random.randint(0, 15),
        })
    return tickets

# Get tool schemas for function calling
tool_schemas = ctx.get_all_tool_schemas()
# Also include the producer itself as a callable tool
tool_schemas.insert(0, {
    "type": "function",
    "function": {
        "name": "load_support_tickets",
        "description": "Load all open support tickets from the helpdesk system.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
})
print(f"Registered {len(tool_schemas)} tools for the agent")

## 2. The Simulated Agent

In [ ]:
def simulate_agent_turn(turn, messages):
    """Scripted agent behavior for demo purposes."""
    wid = None
    for m in messages:
        if m.get("role") == "tool":
            try:
                data = json.loads(m["content"])
                if isinstance(data, dict) and "workspace_id" in data:
                    wid = data["workspace_id"]
            except (json.JSONDecodeError, TypeError):
                pass

    if turn == 0:
        return [{"id": "call_0", "name": "load_support_tickets", "arguments": {}}], None

    if turn == 1 and wid:
        return [{"id": "call_1", "name": "tickets_pipe", "arguments": {
            "workspace_id": wid,
            "steps": [
                {"filter": {"urgency": "critical"}},
                {"sort": {"field": "created_days_ago", "order": "asc"}},
                {"limit": 5},
                {"select": ["ticket_id", "subject", "urgency", "category", "created_days_ago", "assigned_to"]},
            ],
        }}], None

    if turn == 2 and wid:
        return [{"id": "call_2", "name": "tickets_aggregate", "arguments": {
            "workspace_id": wid,
            "group_by": "category",
            "metrics": {"count": "count", "avg_age": "mean:created_days_ago"},
        }}], None

    if turn == 3 and wid:
        return [{"id": "call_3", "name": "tickets_pipe", "arguments": {
            "workspace_id": wid,
            "steps": [
                {"filter": {"urgency": {"$in": ["critical", "high"]}, "created_days_ago": {"$gte": 30}}},
                {"count": True},
            ],
        }}], None

    return [], (
        "Here's the support queue analysis:\n\n"
        "1. **Critical tickets (newest first):** Found the top 5 most recent critical tickets. "
        "They span billing, technical, and bug_report categories.\n\n"
        "2. **Category breakdown:** Technical and billing categories have the highest volumes.\n\n"
        "3. **Aging alert:** There are stale high/critical tickets older than 30 days that need escalation.\n\n"
        "Recommended actions: Prioritize the 30+ day critical tickets and assign more agents to billing."
    )

## 3. The Agent Loop

In [ ]:
def run_agent(user_message: str, max_turns: int = 10):
    """Run a multi-turn agent loop with ctxtual tool handling."""

    messages = [
        {"role": "system", "content": ctx.system_prompt()},
        {"role": "user", "content": user_message},
    ]

    print(f"🧑 User: {user_message}\n")

    for turn in range(max_turns):
        # Get LLM response
        if USE_REAL_API:
            from openai import OpenAI
            client = OpenAI()
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=messages,
                tools=tool_schemas,
                tool_choice="auto",
            )
            msg = response.choices[0].message
            if msg.tool_calls:
                tool_calls = [
                    {"id": tc.id, "name": tc.function.name, "arguments": json.loads(tc.function.arguments)}
                    for tc in msg.tool_calls
                ]
            else:
                tool_calls = []
            text_response = msg.content
        else:
            tool_calls, text_response = simulate_agent_turn(turn, messages)

        # Handle text response (agent is done)
        if text_response and not tool_calls:
            print(f"🤖 Agent: {text_response}\n")
            messages.append({"role": "assistant", "content": text_response})
            break

        # --- Handle tool calls ---
        if tool_calls:
            messages.append({
                "role": "assistant",
                "content": text_response or "",
                "tool_calls": [
                    {"id": tc["id"], "type": "function",
                     "function": {"name": tc["name"], "arguments": json.dumps(tc["arguments"])}}
                    for tc in tool_calls
                ],
            })

            for tc in tool_calls:
                name, args = tc["name"], tc["arguments"]
                print(f"  🔧 Turn {turn}: {name}({json.dumps(args, separators=(',', ':'))[:100]})")

                # Dispatch through ctxtual (or call producer directly)
                if name == "load_support_tickets":
                    result = load_support_tickets()
                else:
                    result = ctx.dispatch_tool_call(name, args)

                result_str = json.dumps(result) if not isinstance(result, str) else result
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc["id"],
                    "content": result_str[:4000],
                })

                # Show compact result
                if isinstance(result, dict) and "workspace_id" in result and "item_count" in result:
                    print(f"       → Workspace ready: {result['item_count']} items")
                elif isinstance(result, dict) and "items" in result:
                    print(f"       → {len(result['items'])} items returned")
                elif isinstance(result, dict) and "count" in result:
                    print(f"       → count: {result['count']}")
                elif isinstance(result, dict) and "groups" in result:
                    print(f"       → {len(result['groups'])} groups")
                else:
                    preview = result_str[:80]
                    print(f"       → {preview}{'...' if len(result_str) > 80 else ''}")

    print(f"\n  📊 Total turns: {turn + 1}, messages: {len(messages)}")
    return messages

In [ ]:
# Run the agent!
messages = run_agent("Analyze our support ticket queue. Find the most critical issues, break down by category, and flag any stale tickets.")

## 4. What the System Prompt Looks Like

In [ ]:
print("=== System Prompt (auto-generated) ===\n")
print(ctx.system_prompt())

## 5. Integration Adapters (OpenAI, Anthropic, LangChain)

In [ ]:
# ctxtual provides zero-dependency adapters for major frameworks

# OpenAI adapter
from ctxtual.integrations.openai import to_openai_tools, handle_tool_calls
tools = to_openai_tools(ctx)
print(f"OpenAI tools: {len(tools)} functions")

# Anthropic adapter
from ctxtual.integrations.anthropic import to_anthropic_tools
tools_anthropic = to_anthropic_tools(ctx)
print(f"Anthropic tools: {len(tools_anthropic)} functions")

# LangChain adapter (requires langchain-core — skip if not installed)
try:
    from ctxtual.integrations.langchain import to_langchain_tools
    lc_tools = to_langchain_tools(ctx)
    print(f"LangChain tools: {len(lc_tools)} tool objects")
    for t in lc_tools[:3]:
        print(f"  - {t.name}: {t.description[:60]}...")
except ImportError:
    print("LangChain adapter: skipped (langchain-core not installed)")

## Key Takeaways

1. **The agent never sees raw data** — only WorkspaceRef with counts, JSON Schema, and available tools
2. **Multi-step exploration in one call** — pipelines replace 4+ round-trips
3. **Error recovery is built in** — wrong workspace_id or tool name returns helpful error dicts
4. **System prompt is auto-generated** — no manual prompt engineering for tool usage
5. **Framework-agnostic** — adapters for OpenAI, Anthropic, LangChain; or use `dispatch_tool_call` directly

**Next:** [05_advanced_patterns.ipynb](05_advanced_patterns.ipynb) — multi-workspace, mutations, concurrency, custom tools